# 1 — Dataset audit

# Inspect the files using Python

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/raw")

files = [
    "train.csv",
    "test.csv",
    "stores.csv",
    "oil.csv",
    "holidays_events.csv",
    "transactions.csv",
    "sample_submission.csv",
]

missing_files = [file for file in files if not (DATA_DIR / file).exists()]

if missing_files:
    raise FileNotFoundError(f"Missing files: {missing_files}")

for file in files:
    df = pd.read_csv(DATA_DIR / file, nrows=5)

    print(f"\n{'=' * 60}")
    print(f"FILE: {file}")
    print(f"COLUMNS: {list(df.columns)}")
    print(df.head(3).to_string(index=False))


FILE: train.csv
COLUMNS: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']
 id       date  store_nbr     family  sales  onpromotion
  0 2013-01-01          1 AUTOMOTIVE    0.0            0
  1 2013-01-01          1  BABY CARE    0.0            0
  2 2013-01-01          1     BEAUTY    0.0            0

FILE: test.csv
COLUMNS: ['id', 'date', 'store_nbr', 'family', 'onpromotion']
     id       date  store_nbr     family  onpromotion
3000888 2017-08-16          1 AUTOMOTIVE            0
3000889 2017-08-16          1  BABY CARE            0
3000890 2017-08-16          1     BEAUTY            2

FILE: stores.csv
COLUMNS: ['store_nbr', 'city', 'state', 'type', 'cluster']
 store_nbr  city     state type  cluster
         1 Quito Pichincha    D       13
         2 Quito Pichincha    D       13
         3 Quito Pichincha    D        8

FILE: oil.csv
COLUMNS: ['date', 'dcoilwtico']
      date  dcoilwtico
2013-01-01         NaN
2013-01-02       93.14
2013-01-03       92.97

FILE: hol

# Ask questions

Does each date/store/product-family combination occur only once?



Are any expected combinations missing?



Are there missing or invalid values?



Does the training data cover the history we expect?



Do the stores in the sales data exist in stores.csv?

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/raw")

train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")

# 1. Dataset sizes
print("TRAIN SHAPE:", train.shape)
print("TEST SHAPE:", test.shape)
print("STORES SHAPE:", stores.shape)

# 2. Historical coverage
print("\nTRAIN DATE RANGE:")
print(train["date"].min(), "to", train["date"].max())

print("\nTEST DATE RANGE:")
print(test["date"].min(), "to", test["date"].max())

print("\nUNIQUE COUNTS:")
print("Stores:", train["store_nbr"].nunique())
print("Product families:", train["family"].nunique())
print("Training dates:", train["date"].nunique())

# 3. Check the intended observation grain
key = ["date", "store_nbr", "family"]

print("\nDUPLICATE OBSERVATIONS:")
print("Train:", train.duplicated(key).sum())
print("Test:", test.duplicated(key).sum())

print("\nDUPLICATE IDS:")
print("Train:", train["id"].duplicated().sum())
print("Test:", test["id"].duplicated().sum())

# 4. Missing values
print("\nMISSING VALUES IN TRAIN:")
print(train.isna().sum())

# 5. Sales and promotion distributions
print("\nSALES SUMMARY:")
print(train["sales"].describe())

print("\nPROMOTION SUMMARY:")
print(train["onpromotion"].describe())

print(
    "Fraction of training rows with zero sales:",
    (train["sales"] == 0).mean()
)

print(
    "Fraction of training rows with promotions:",
    (train["onpromotion"] > 0).mean()
)

# 6. Check store metadata coverage
print("\nSTORE METADATA:")
print("Duplicate store numbers:", stores["store_nbr"].duplicated().sum())
print(
    "Sales rows with unknown store:",
    (~train["store_nbr"].isin(stores["store_nbr"])).sum()
)

TRAIN SHAPE: (3000888, 6)
TEST SHAPE: (28512, 5)
STORES SHAPE: (54, 5)

TRAIN DATE RANGE:
2013-01-01 00:00:00 to 2017-08-15 00:00:00

TEST DATE RANGE:
2017-08-16 00:00:00 to 2017-08-31 00:00:00

UNIQUE COUNTS:
Stores: 54
Product families: 33
Training dates: 1684

DUPLICATE OBSERVATIONS:
Train: 0
Test: 0

DUPLICATE IDS:
Train: 0
Test: 0

MISSING VALUES IN TRAIN:
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

SALES SUMMARY:
count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

PROMOTION SUMMARY:
count    3.000888e+06
mean     2.602770e+00
std      1.221888e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.410000e+02
Name: onpromotion, dtype: float64
Fraction of training rows with zero sales: 0.3129506999261552
Fraction 

## Test A: Verify the expected number of records

In [3]:
# Expected rows if every date has every store-family combination
expected_train_rows = (
    train["date"].nunique()
    * train["store_nbr"].nunique()
    * train["family"].nunique()
)

expected_test_rows = (
    test["date"].nunique()
    * test["store_nbr"].nunique()
    * test["family"].nunique()
)

print("Expected train rows:", expected_train_rows)
print("Actual train rows:", len(train))

print("Expected test rows:", expected_test_rows)
print("Actual test rows:", len(test))

# Verify records per date
rows_per_date = train.groupby("date").size()

print("\nExpected rows per training date:",
      train["store_nbr"].nunique() * train["family"].nunique())

print("Minimum rows per date:", rows_per_date.min())
print("Maximum rows per date:", rows_per_date.max())

assert len(train) == expected_train_rows
assert len(test) == expected_test_rows
assert rows_per_date.nunique() == 1

print("\nPanel integrity checks passed.")

Expected train rows: 3000888
Actual train rows: 3000888
Expected test rows: 28512
Actual test rows: 28512

Expected rows per training date: 1782
Minimum rows per date: 1782
Maximum rows per date: 1782

Panel integrity checks passed.


## Test B: Investigate the zero-sales pattern

In [4]:
# Overall zero-sales share
print("Overall zero-sales share:",
      (train["sales"] == 0).mean())

# Zero-sales share by product family
family_summary = (
    train.groupby("family")
    .agg(
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
        zero_sales_share=(
            "sales",
            lambda s: (s == 0).mean()
        )
    )
    .sort_values("zero_sales_share", ascending=False)
)

print("\nFamilies with the highest zero-sales share:")
print(family_summary.head(10).round(3))

# Zero-sales share by date
daily_summary = (
    train.groupby("date")
    .agg(
        total_sales=("sales", "sum"),
        zero_sales_share=(
            "sales",
            lambda s: (s == 0).mean()
        )
    )
    .sort_values("zero_sales_share", ascending=False)
)

print("\nDates with the highest zero-sales share:")
print(daily_summary.head(10).round(3))

# Inspect one date with a very high zero-sales share
inspect_date = daily_summary.index[0]

print(f"\nRecords for {inspect_date.date()}:")
print(
    train.loc[
        train["date"] == inspect_date,
        ["store_nbr", "family", "sales", "onpromotion"]
    ].head(15)
)

Overall zero-sales share: 0.3129506999261552

Families with the highest zero-sales share:
                            mean_sales  median_sales  zero_sales_share
family                                                                
BOOKS                            0.071           0.0             0.970
BABY CARE                        0.111           0.0             0.941
SCHOOL AND OFFICE SUPPLIES       2.962           0.0             0.741
HOME APPLIANCES                  0.457           0.0             0.735
LADIESWEAR                       7.161           0.0             0.598
MAGAZINES                        2.929           0.0             0.565
PET SUPPLIES                     3.921           0.0             0.546
HARDWARE                         1.138           1.0             0.479
LAWN AND GARDEN                  6.035           1.0             0.468
PLAYERS AND ELECTRONICS          6.187           2.0             0.449

Dates with the highest zero-sales share:
            tota

Books have zero sales in approximately 97% of records, while baby care has zero sales in approximately 94%.


January 1 is a recurring special date


Some Sundays in early 2013 also have high zero-sales shares

## Test A — Are the early-2013 drops related to Sundays?

In [5]:
train["day_of_week"] = train["date"].dt.day_name()

weekday_summary = (
    train.groupby("day_of_week")
    .agg(
        mean_sales=("sales", "mean"),
        zero_sales_share=("sales", lambda s: (s == 0).mean())
    )
)

weekday_summary = weekday_summary.reindex([
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
])

print(weekday_summary.round(3))

             mean_sales  zero_sales_share
day_of_week                              
Monday          346.545             0.310
Tuesday         319.824             0.315
Wednesday       332.909             0.315
Thursday        283.541             0.317
Friday          325.238             0.316
Saturday        433.336             0.297
Sunday          463.085             0.320


## Test B — Examine the specific dates

In [7]:
dates_to_check = pd.to_datetime([
    "2013-01-01",
    "2013-01-20",
    "2013-01-27",
    "2013-02-16",
    "2013-02-17",
    "2013-02-24",
    "2014-01-01",
    "2015-01-01",
    "2016-01-01",
    "2017-01-01",
])

date_summary = (
    train[train["date"].isin(dates_to_check)]
    .groupby("date")
    .agg(
        total_sales=("sales", "sum"),
        zero_sales_share=("sales", lambda s: (s == 0).mean()),
        mean_promotions=("onpromotion", "mean"),
        stores_with_positive_sales=(
            "sales",
            lambda s: s.groupby(
                train.loc[s.index, "store_nbr"]
            ).sum().gt(0).sum()
        )
    )
)

date_summary.round(3)

,total_sales,zero_sales_share,mean_promotions,stores_with_positive_sales
date,,,,
2013-01-01,2511.619,0.990,0.000,1
2013-01-20,461092.386,0.536,0.000,46
2013-01-27,430411.991,0.544,0.000,46
2013-02-16,485660.494,0.539,0.000,46
2013-02-17,399588.910,0.543,0.000,46
2013-02-24,452726.126,0.538,0.000,46
2014-01-01,8602.065,0.972,0.000,2
2015-01-01,12773.617,0.987,0.007,1
2016-01-01,16433.394,0.984,0.089,1


## Test C — Check the holiday calendar

In [8]:
holidays = pd.read_csv(
    DATA_DIR / "holidays_events.csv",
    parse_dates=["date"]
)

holiday_check = holidays[
    holidays["date"].isin(dates_to_check)
].sort_values(["date", "locale", "locale_name"])

print(
    holiday_check[
        ["date", "type", "locale", "locale_name",
         "description", "transferred"]
    ].to_string(index=False)
)

      date    type   locale locale_name        description  transferred
2013-01-01 Holiday National     Ecuador Primer dia del ano        False
2014-01-01 Holiday National     Ecuador Primer dia del ano        False
2015-01-01 Holiday National     Ecuador Primer dia del ano        False
2016-01-01 Holiday National     Ecuador Primer dia del ano        False
2017-01-01 Holiday National     Ecuador Primer dia del ano         True


## Investigate store-level zero-sales patterns

In [9]:
# Total sales per store on each date
store_daily_sales = (
    train.groupby(["date", "store_nbr"], as_index=False)
    ["sales"].sum()
)

# Stores with no sales at all on each date
zero_store_counts = (
    store_daily_sales.assign(
        no_sales=lambda df: df["sales"].eq(0)
    )
    .groupby("date")["no_sales"]
    .sum()
)

print("Dates with the most stores reporting zero total sales:")
print(zero_store_counts.sort_values(ascending=False).head(15))

# First date each store has positive sales
first_positive_sales = (
    store_daily_sales[store_daily_sales["sales"] > 0]
    .groupby("store_nbr")["date"]
    .min()
    .sort_values()
)

print("\nFirst date with positive sales by store:")
print(first_positive_sales)

Dates with the most stores reporting zero total sales:
date
2013-01-01    53
2016-01-01    53
2017-01-01    53
2015-01-01    53
2014-01-01    52
2013-07-15     8
2013-01-02     8
2013-07-08     8
2013-07-09     8
2013-07-11     8
2013-07-12     8
2013-07-13     8
2013-07-14     8
2013-07-16     8
2013-05-07     8
Name: no_sales, dtype: int64

First date with positive sales by store:
store_nbr
25   2013-01-01
28   2013-01-02
30   2013-01-02
31   2013-01-02
32   2013-01-02
33   2013-01-02
34   2013-01-02
35   2013-01-02
37   2013-01-02
38   2013-01-02
39   2013-01-02
1    2013-01-02
41   2013-01-02
43   2013-01-02
44   2013-01-02
45   2013-01-02
46   2013-01-02
47   2013-01-02
48   2013-01-02
49   2013-01-02
50   2013-01-02
51   2013-01-02
26   2013-01-02
40   2013-01-02
27   2013-01-02
23   2013-01-02
2    2013-01-02
3    2013-01-02
4    2013-01-02
5    2013-01-02
6    2013-01-02
7    2013-01-02
8    2013-01-02
9    2013-01-02
10   2013-01-02
11   2013-01-02
24   2013-01-02
13   2013-01

## Investigate late-starting stores

In [10]:
# Identify stores whose first positive sales occurred after Jan 2013
late_starts = first_positive_sales[
    first_positive_sales > pd.Timestamp("2013-01-02")
].rename("first_positive_date")

late_starts = (
    late_starts.reset_index()
    .merge(stores, on="store_nbr", how="left")
    .sort_values("first_positive_date")
)

print(late_starts.to_string(index=False))

 store_nbr first_positive_date          city                          state type  cluster
        36          2013-05-09      Libertad                         Guayas    E       10
        53          2014-05-29         Manta                         Manabi    D       13
        20          2015-02-13         Quito                      Pichincha    B        6
        29          2015-03-20     Guayaquil                         Guayas    E       10
        21          2015-07-24 Santo Domingo Santo Domingo de los Tsachilas    B        6
        42          2015-08-21        Cuenca                          Azuay    D        2
        22          2015-10-09          Puyo                        Pastaza    C        7
        52          2017-04-20         Manta                         Manabi    A       11


In [11]:
# Aggregate sales across all product families for each store and date
store_daily_sales = (
    train.groupby(["date", "store_nbr"], as_index=False)["sales"]
    .sum()
)

for store_id in late_starts["store_nbr"]:
    first_date = late_starts.loc[
        late_starts["store_nbr"] == store_id,
        "first_positive_date"
    ].iloc[0]

    history = store_daily_sales[
        (store_daily_sales["store_nbr"] == store_id)
        & (store_daily_sales["date"].between(
            first_date - pd.Timedelta(days=7),
            first_date + pd.Timedelta(days=7)
        ))
    ]

    print(f"\nStore {store_id} — first positive date: {first_date.date()}")
    print(history.to_string(index=False))


Store 36 — first positive date: 2013-05-09
      date  store_nbr        sales
2013-05-02         36     0.000000
2013-05-03         36     0.000000
2013-05-04         36     0.000000
2013-05-05         36     0.000000
2013-05-06         36     0.000000
2013-05-07         36     0.000000
2013-05-08         36     0.000000
2013-05-09         36 14557.989000
2013-05-10         36  8884.132000
2013-05-11         36 11565.197002
2013-05-12         36 11277.695000
2013-05-13         36  7542.378990
2013-05-14         36  6464.965999
2013-05-15         36  5926.713996
2013-05-16         36  5711.977997

Store 53 — first positive date: 2014-05-29
      date  store_nbr     sales
2014-05-22         53     0.000
2014-05-23         53     0.000
2014-05-24         53     0.000
2014-05-25         53     0.000
2014-05-26         53     0.000
2014-05-27         53     0.000
2014-05-28         53     0.000
2014-05-29         53 12187.001
2014-05-30         53  6775.058
2014-05-31         53 12095.131


# Milestone 1 — Dataset audit: what we learned

- January 1: 52–53 of the 54 stores report zero total sales. This is consistent with a widespread holiday closure pattern, rather than ordinary demand.
- A small number of stores begin recording positive sales later: Store 36 first shows positive sales in May 2013, while several stores first show positive sales in 2014–2017. This could indicate later openings, but first positive sales alone doesn't prove an opening date.
- Assumption -Before a store's first positive-sales date, treat its recorded zeros as pre-opening inactivity. Preserve the original data, document the assumption, and revisit it if business evidence becomes available.

# 2 — Build a baseline forecast

We'll compare two baselines:

No machine-learning model is being trained yet. We're using two rule-based forecasting methods:
- Last-value baseline: predict sales using the last observed value for each store–family combination.
- Seasonal-naive baseline: predict sales using the latest observed value for the same weekday.

In [13]:
import numpy as np

# 1. Chronological split: final 16 days of training data
validation_start = train["date"].max() - pd.Timedelta(days=15)
cutoff_date = validation_start - pd.Timedelta(days=1)

history = train[train["date"] <= cutoff_date].copy()
validation = train[train["date"] >= validation_start].copy()

print("History:", history["date"].min(), "to", history["date"].max())
print("Validation:", validation["date"].min(), "to", validation["date"].max())
print("Validation rows:", len(validation))

# 2. Baseline A: last observed sales for each store-family
last_value = (
    history.sort_values("date")
    .groupby(["store_nbr", "family"], as_index=False)
    .tail(1)[["store_nbr", "family", "sales"]]
    .rename(columns={"sales": "pred_last_value"})
)

validation = validation.merge(
    last_value,
    on=["store_nbr", "family"],
    how="left",
    validate="many_to_one"
)

# 3. Baseline B: latest observed sales for the same weekday
history["day_of_week"] = history["date"].dt.dayofweek
validation["day_of_week"] = validation["date"].dt.dayofweek

recent_week = history[
    history["date"] > cutoff_date - pd.Timedelta(days=7)
][["store_nbr", "family", "day_of_week", "sales"]].copy()

recent_week = recent_week.rename(
    columns={"sales": "pred_seasonal_naive"}
)

validation = validation.merge(
    recent_week,
    on=["store_nbr", "family", "day_of_week"],
    how="left",
    validate="many_to_one"
)

# 4. Metrics
def evaluate(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    wape = (
        np.abs(y_true - y_pred).sum()
        / np.abs(y_true).sum()
        * 100
    )
    return {"MAE": mae, "WAPE (%)": wape}

for name, pred_col in [
    ("Last value", "pred_last_value"),
    ("Seasonal naive", "pred_seasonal_naive"),
]:
    print(name, evaluate(validation["sales"], validation[pred_col]))

# 5. Confirm there are no missing predictions
print("\nMissing predictions:")
print(validation[["pred_last_value", "pred_seasonal_naive"]].isna().sum())


History: 2013-01-01 00:00:00 to 2017-07-30 00:00:00
Validation: 2017-07-31 00:00:00 to 2017-08-15 00:00:00
Validation rows: 28512
Last value {'MAE': 207.50578961580388, 'WAPE (%)': 44.420190811432654}
Seasonal naive {'MAE': 96.53539090179574, 'WAPE (%)': 20.665064294608086}

Missing predictions:
pred_last_value        0
pred_seasonal_naive    0
dtype: int64


Milestone 2 — Baselines

-    Validation period: July 31–August 15, 2017
-    Metric	Last value	Seasonal naive
-    MAE	207.51	96.54
-    WAPE	44.42%	20.67%
-    Missing predictions	0	0
-       Benchmark to beat: seasonal-naive WAPE of 20.67%.


We will keep the validation dates and metric definitions fixed when evaluating LightGBM, so comparisons remain fair.

# 3 — Build our first ML forecasting model

Retain recursive forecasting for the 7-, 14-, and 30-day horizons, with one LightGBM model per store and family as a categorical feature

Goal: Train a LightGBM model for one store, predict daily sales across its 33 product families, and compare it with the seasonal-naive baseline.

In [16]:

# If needed, install in a separate notebook cell:
# %pip install lightgbm

import lightgbm as lgb

print("LightGBM version:", lgb.__version__)


LightGBM version: 4.5.0


In [17]:
# FEATURE ENGINEERING

# Fixed experiment settings
STORE_ID = 1
VALIDATION_DAYS = 16

validation_start = train["date"].max() - pd.Timedelta(
    days=VALIDATION_DAYS - 1
)
cutoff_date = validation_start - pd.Timedelta(days=1)

# Keep one store for the first experiment
store_data = (
    train[train["store_nbr"] == STORE_ID]
    .sort_values(["family", "date"])
    .copy()
)

# Build lag and rolling features using past sales only
def make_features(df):
    df = df.sort_values(["family", "date"]).copy()
    group = df.groupby("family", observed=True)["sales"]

    for lag in [1, 7, 14]:
        df[f"lag_{lag}"] = group.shift(lag)

    for window in [7, 28]:
        df[f"rolling_mean_{window}"] = (
            df.groupby("family", observed=True)["sales"]
            .transform(lambda s: s.shift(1).rolling(window).mean())
        )

    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["month"] = df["date"].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype("int8")

    return df

# Feature creation uses each family's historical sequence
feature_data = make_features(store_data)

# Add static metadata
feature_data = feature_data.merge(
    stores[["store_nbr", "type", "cluster", "city"]],
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

# Split by date after creating lag features
model_train = feature_data[
    feature_data["date"] <= cutoff_date
].dropna(
    subset=[
        "lag_1", "lag_7", "lag_14",
        "rolling_mean_7", "rolling_mean_28"
    ]
).copy()

model_valid = feature_data[
    feature_data["date"] >= validation_start
].copy()

print("Store:", STORE_ID)
print("Training rows:", len(model_train))
print("Validation rows:", len(model_valid))
print("Product families:", store_data["family"].nunique())
print("Training dates:", model_train["date"].min(), "to", model_train["date"].max())
print("Validation dates:", model_valid["date"].min(), "to", model_valid["date"].max())


Store: 1
Training rows: 54120
Validation rows: 528
Product families: 33
Training dates: 2013-01-29 00:00:00 to 2017-07-30 00:00:00
Validation dates: 2017-07-31 00:00:00 to 2017-08-15 00:00:00


### Understand clusters for stores

In [18]:
cluster_summary = (
    stores.groupby("cluster")
    .agg(
        n_stores=("store_nbr", "nunique"),
        n_store_types=("type", "nunique"),
        cities=("city", "nunique"),
        states=("state", "nunique"),
    )
    .sort_index()
)

print(cluster_summary)
print("\nStores per cluster:")
print(stores.groupby("cluster")["store_nbr"].apply(list))

         n_stores  n_store_types  cities  states
cluster                                         
1               3              1       3       2
2               2              1       1       1
3               7              1       6       5
4               3              1       3       3
5               1              1       1       1
6               6              1       5       4
7               2              1       2       2
8               3              1       1       1
9               2              1       2       2
10              6              3       4       3
11              3              1       2       2
12              1              1       1       1
13              4              1       2       2
14              4              1       2       2
15              5              1       4       4
16              1              1       1       1
17              1              1       1       1

Stores per cluster:
cluster
1                     [24, 25, 27]
2    

In [21]:

import numpy as np
import pandas as pd

# 1. Combine train and test into one base table
master_df = pd.concat(
    [
        train.assign(is_train=True),
        test.assign(sales=np.nan, is_train=False),
    ],
    ignore_index=True,
)

# 2. Add store metadata
master_df = master_df.merge(
    stores[["store_nbr", "city", "state", "type", "cluster"]],
    on="store_nbr",
    how="left",
    validate="many_to_one",
)


from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/raw")

train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")
oil = pd.read_csv(DATA_DIR / "oil.csv", parse_dates=["date"])
holidays_events = pd.read_csv(
    DATA_DIR / "holidays_events.csv", parse_dates=["date"]
)
transactions = pd.read_csv(
    DATA_DIR / "transactions.csv", parse_dates=["date"]
)

print("Loaded datasets:")
for name, df in {
    "train": train,
    "test": test,
    "stores": stores,
    "oil": oil,
    "holidays_events": holidays_events,
    "transactions": transactions,
}.items():
    print(f"{name}: {df.shape}")


# 3. Summarize holiday events by date before merging
holiday_daily = (
    holidays_events.assign(
        is_national=holidays_events["locale"].eq("National"),
        is_regional=holidays_events["locale"].eq("Regional"),
        is_local=holidays_events["locale"].eq("Local"),
        is_transferred=(
            holidays_events["transferred"]
            .astype(str).str.lower().eq("true")
        ),
    )
    .groupby("date", as_index=False)
    .agg(
        holiday_event_count=("type", "size"),
        has_national_event=("is_national", "max"),
        has_regional_event=("is_regional", "max"),
        has_local_event=("is_local", "max"),
        has_transferred_event=("is_transferred", "max"),
    )
)

master_df = master_df.merge(
    holiday_daily,
    on="date",
    how="left",
    validate="many_to_one",
)

# 4. Add oil prices
master_df = master_df.merge(
    oil,
    on="date",
    how="left",
    validate="many_to_one",
)

# 5. Add store transactions
master_df = master_df.merge(
    transactions,
    on=["date", "store_nbr"],
    how="left",
    validate="many_to_one",
)

# 6. Add calendar features
master_df["day_of_week"] = master_df["date"].dt.dayofweek
master_df["day_of_month"] = master_df["date"].dt.day
master_df["month"] = master_df["date"].dt.month
master_df["year"] = master_df["date"].dt.year
master_df["is_weekend"] = (
    master_df["day_of_week"] >= 5
).astype("int8")

# 7. Inspect the result
print("Master shape:", master_df.shape)
print("Date range:", master_df["date"].min(), "to", master_df["date"].max())
print("Training rows:", master_df["is_train"].sum())
print("Test rows:", (~master_df["is_train"]).sum())

print("\nMissing values:")
print(
    master_df.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nDuplicate date-store-family rows:",
      master_df.duplicated(["date", "store_nbr", "family"]).sum())

print("\nColumns:")
print(master_df.columns.tolist())


Loaded datasets:
train: (3000888, 6)
test: (28512, 5)
stores: (54, 5)
oil: (1218, 2)
holidays_events: (350, 6)
transactions: (83488, 3)
Master shape: (3029400, 23)
Date range: 2013-01-01 00:00:00 to 2017-08-31 00:00:00
Training rows: 3000888
Test rows: 28512

Missing values:
holiday_event_count      2578554
has_transferred_event    2578554
has_local_event          2578554
has_regional_event       2578554
has_national_event       2578554
dcoilwtico                935550
transactions              274296
sales                      28512
id                             0
year                           0
month                          0
day_of_month                   0
cluster                        0
date                           0
type                           0
dtype: int64

Duplicate date-store-family rows: 0

Columns:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'day_of_week', 'is_train', 'city', 'state', 'type', 'cluster', 'holiday_event_count', 'has_national_event',

In [22]:
# EDA: Explore the combined dataset

# 1. Target distribution
tr = master_df.loc[master_df["is_train"]].copy()

print("=== TARGET DISTRIBUTION ===")
print(tr["sales"].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]))
print("Zero-sales share:", round(tr["sales"].eq(0).mean() * 100, 2), "%")
print("Negative sales:", tr["sales"].lt(0).sum())

# 2. Product-family behavior
family_eda = (
    tr.groupby("family")
    .agg(
        rows=("sales", "size"),
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
        zero_share=("sales", lambda s: s.eq(0).mean()),
        total_sales=("sales", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)
print("\n=== FAMILY SUMMARY ===")
print(family_eda.round(3).to_string())

# 3. Store behavior
store_eda = (
    tr.groupby(["store_nbr", "cluster"])
    .agg(
        mean_sales=("sales", "mean"),
        total_sales=("sales", "sum"),
        zero_share=("sales", lambda s: s.eq(0).mean()),
    )
    .sort_values("total_sales", ascending=False)
)
print("\n=== STORE SUMMARY ===")
print(store_eda.round(2).to_string())

# 4. Weekly aggregate behavior
weekly = (
    tr.groupby(pd.Grouper(key="date", freq="W"))["sales"]
    .sum()
)
print("\n=== WEEKLY SALES ===")
print(weekly.describe().round(2))
print("Lowest-sales weeks:")
print(weekly.nsmallest(5))
print("Highest-sales weeks:")
print(weekly.nlargest(5))

# 5. Feature coverage, separated by train and test
coverage_cols = [
    "onpromotion", "dcoilwtico", "transactions",
    "holiday_event_count", "has_national_event",
    "has_regional_event", "has_local_event",
    "has_transferred_event",
]
print("\n=== MISSINGNESS BY SPLIT (%) ===")
print(
    master_df.groupby("is_train")[coverage_cols]
    .apply(lambda x: x.isna().mean().mul(100))
    .round(2)
    .rename(index={True: "train", False: "test"})
    .to_string()
)

# 6. Check whether missing transaction rows correspond to missing dates
transaction_coverage = (
    master_df.groupby(["is_train", "date"])["transactions"]
    .apply(lambda s: s.notna().sum())
    .groupby(level=0)
    .agg(["min", "median", "max"])
)
print("\n=== STORES WITH TRANSACTION DATA PER DATE ===")
print(transaction_coverage)

# 7. Promotion behavior by family
promotion_eda = (
    tr.groupby("family")
    .agg(
        promotion_mean=("onpromotion", "mean"),
        promotion_median=("onpromotion", "median"),
        promotion_max=("onpromotion", "max"),
    )
    .sort_values("promotion_mean", ascending=False)
)
print("\n=== PROMOTIONS BY FAMILY ===")
print(promotion_eda.round(2).to_string())


=== TARGET DISTRIBUTION ===
count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
1%       0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
95%      1.965000e+03
99%      5.507000e+03
max      1.247170e+05
Name: sales, dtype: float64
Zero-sales share: 31.3 %
Negative sales: 0

=== FAMILY SUMMARY ===
                             rows  mean_sales  median_sales  zero_share   total_sales
family                                                                               
GROCERY I                   90936    3776.972      3185.000       0.081  3.434627e+08
BEVERAGES                   90936    2385.793      1784.000       0.081  2.169545e+08
PRODUCE                     90936    1349.352       398.290       0.284  1.227047e+08
CLEANING                    90936    1072.417       938.000       0.081  9.752129e+07
DAIRY                       90936     709.155       520.000       0.081  6.448771e+07
BREAD/BAKERY                90936

Exclude transactions from the first model. It's entirely missing in the test period, so using it would create a deployment problem. We can test a separate historical-transaction feature later if we can generate valid future estimates.



Keep promotions and calendar features. Promotions are known for the competition test dates, and calendar effects are available in production. For future scenarios, promotion assumptions must be explicit.



Treat oil as an experiment, not a guaranteed feature. Its missingness is substantial, and future prices aren't known. We'll establish a robust baseline without it, then test whether historical oil information adds value.

In [25]:

# Build the modeling table from training observations only.
# Keep master_df unchanged as our canonical source table.

feature_cols_base = [
    "id", "date", "store_nbr", "family", "sales",
    "onpromotion", "city", "state", "type", "cluster",
    "day_of_week", "day_of_month", "month", "year",
    "is_weekend", "holiday_event_count",
    "has_national_event", "has_regional_event",
    "has_local_event", "has_transferred_event",
]

model_df = (
    master_df.loc[master_df["is_train"], feature_cols_base]
    .copy()
    .sort_values(["store_nbr", "family", "date"])
    .reset_index(drop=True)
)

# No holiday event on a date is represented as 0.
holiday_cols = [
    "holiday_event_count",
    "has_national_event",
    "has_regional_event",
    "has_local_event",
    "has_transferred_event",
]
model_df[holiday_cols] = model_df[holiday_cols].fillna(0)

# Historical sales features: shift first to prevent target leakage.
series_keys = ["store_nbr", "family"]
sales_group = model_df.groupby(series_keys, observed=True)["sales"]

for lag in [1, 7, 14, 28, 364]:
    model_df[f"lag_{lag}"] = sales_group.shift(lag)

for window in [7, 28, 56]:
    model_df[f"rolling_mean_{window}"] = (
        model_df.groupby(series_keys, observed=True)["sales"]
        .transform(
            lambda s: s.shift(1).rolling(
                window=window, min_periods=window
            ).mean()
        )
    )

model_df["rolling_std_7"] = (
    model_df.groupby(series_keys, observed=True)["sales"]
    .transform(
        lambda s: s.shift(1).rolling(
            window=7, min_periods=7
        ).std()
    )
)

lag_cols = [
    "lag_1", "lag_7", "lag_14", "lag_28", "lag_364",
    "rolling_mean_7", "rolling_mean_28",
    "rolling_mean_56", "rolling_std_7",
]

# Explicit categorical columns for LightGBM.
for col in ["store_nbr", "family", "city", "state", "type", "cluster"]:
    model_df[col] = model_df[col].astype("category")

print("Feature table:", model_df.shape)
print("\nMissing engineered features (%):")
print((model_df[lag_cols].isna().mean() * 100).round(2))

print("\nFeature preview:")
print(
    model_df[
        ["date", "store_nbr", "family", "sales",
         "onpromotion"] + lag_cols[:5]
    ].tail(10).to_string(index=False)
)


Feature table: (3000888, 29)

Missing engineered features (%):
lag_1               0.06
lag_7               0.42
lag_14              0.83
lag_28              1.66
lag_364            21.62
rolling_mean_7      0.42
rolling_mean_28     1.66
rolling_mean_56     3.33
rolling_std_7       0.42
dtype: float64

Feature preview:
      date store_nbr  family  sales  onpromotion  lag_1  lag_7  lag_14  lag_28  lag_364
2017-08-06        54 SEAFOOD    0.0            0    3.0    4.0     5.0     6.0      3.0
2017-08-07        54 SEAFOOD    0.0            0    0.0    4.0     1.0    10.0      5.0
2017-08-08        54 SEAFOOD   12.0            0    0.0    3.0     7.0     3.0      1.0
2017-08-09        54 SEAFOOD    4.0            0   12.0    3.0     3.0     1.0      1.0
2017-08-10        54 SEAFOOD    2.0            0    4.0    5.0     2.0     0.0      0.0
2017-08-11        54 SEAFOOD    0.0            0    2.0    0.0     4.0     0.0      2.0
2017-08-12        54 SEAFOOD    1.0            1    0.0    3.0 

In [28]:

# STEP 2A — Prepare and audit model features

import numpy as np
import pandas as pd

holiday_flag_cols = [
    "has_national_event",
    "has_regional_event",
    "has_local_event",
    "has_transferred_event",
]

# 1. Normalize holiday flags safely
flag_map = {
    "true": 1,
    "false": 0,
    "1": 1,
    "0": 0,
}

for col in holiday_flag_cols:
    normalized = model_df[col].astype("string").str.strip().str.lower()
    unknown = normalized.dropna()[~normalized.dropna().isin(flag_map)]

    if not unknown.empty:
        raise ValueError(
            f"Unexpected values in {col}: {unknown.unique()[:10]}"
        )

    model_df[col] = (
        normalized.map(flag_map)
        .fillna(0)
        .astype("int8")
    )

# Missing event count means no recorded event for that date
model_df["holiday_event_count"] = (
    pd.to_numeric(model_df["holiday_event_count"], errors="raise")
    .fillna(0)
    .astype("int16")
)

# 2. Explicit categorical encoding
categorical_cols = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type",
    "cluster",
]

for col in categorical_cols:
    model_df[col] = model_df[col].astype("category")

# 3. Confirm the dataset is valid
assert model_df["sales"].notna().all(), "Training target has missing values"
assert (model_df["sales"] >= 0).all(), "Unexpected negative sales"
assert not model_df.duplicated(["date", "store_nbr", "family"]).any()

unexpected_objects = model_df.select_dtypes(include=["object"]).columns.tolist()
assert not unexpected_objects, f"Object columns remain: {unexpected_objects}"

print("Shape:", model_df.shape)
print("\nDtypes:")
print(model_df.dtypes.to_string())

print("\nMissing-value percentage:")
print(
    (model_df.isna().mean().mul(100))
    .loc[lambda s: s.gt(0)]
    .sort_values(ascending=False)
    .round(2)
    .to_string()
)

print("\nCategorical cardinality:")
for col in categorical_cols:
    print(f"{col}: {model_df[col].nunique()} categories")

print("\nTarget rows:", len(model_df))
print("Date range:", model_df["date"].min(), "to", model_df["date"].max())


Shape: (3000888, 29)

Dtypes:
id                                int64
date                     datetime64[ns]
store_nbr                      category
family                         category
sales                           float64
onpromotion                       int64
city                           category
state                          category
type                           category
cluster                        category
day_of_week                       int32
day_of_month                      int32
month                             int32
year                              int32
is_weekend                         int8
holiday_event_count               int16
has_national_event                 int8
has_regional_event                 int8
has_local_event                    int8
has_transferred_event              int8
lag_1                           float64
lag_7                           float64
lag_14                          float64
lag_28                          float64
lag_364   

### Hyperparameter tuning strategy

3A. Select five stores
Choose a reproducible random sample with a fixed seed. We'll report the selected stores and check their sales volume and zero-sales share, so we know whether the sample is unusually easy or difficult.




3B. Use three chronological folds
Each fold uses only data before its validation window. We'll use three successive 14-day windows ending before July 31, 2017, keeping your existing July 31–August 15 holdout untouched for the final comparison.




3C. Run 20 randomized configurations
Search over learning rate, number of leaves, tree count, minimum leaf size, sampling fractions and regularization. This means 60 model fits across the three folds—not 20 fits.




3D. Score honestly
Use recursive multi-day predictions: once a forecasted day is predicted, its prediction—not its actual sales—is used to construct the next day's lags. Select parameters using average fold WAPE, while also recording MAE and comparing against seasonal-naive forecasts.

In [29]:

# STEP 4A — Select stores and define chronological folds
SEED = 42
HORIZON = 14

# Reproducibly select five stores
store_summary = (
    model_df.groupby("store_nbr", observed=True)
    .agg(
        mean_sales=("sales", "mean"),
        total_sales=("sales", "sum"),
        zero_sales_share=("sales", lambda s: s.eq(0).mean()),
        n_rows=("sales", "size"),
    )
    .reset_index()
)

selected_stores = (
    store_summary["store_nbr"]
    .sample(n=5, random_state=SEED)
    .tolist()
)

print("Selected stores:", selected_stores)

print(
    store_summary.loc[
        store_summary["store_nbr"].isin(selected_stores)
    ].sort_values("store_nbr").to_string(index=False)
)

# Three non-overlapping 14-day validation windows.
# Every fold's training data ends before its validation window.
folds = [
    ("fold_1", pd.Timestamp("2017-06-19"), pd.Timestamp("2017-07-02")),
    ("fold_2", pd.Timestamp("2017-07-03"), pd.Timestamp("2017-07-16")),
    ("fold_3", pd.Timestamp("2017-07-17"), pd.Timestamp("2017-07-30")),
]

for name, start, end in folds:
    train_end = start - pd.Timedelta(days=1)
    print(
        f"{name}: train through {train_end.date()}, "
        f"validate {start.date()} to {end.date()}"
    )


Selected stores: [20, 50, 49, 13, 45]
store_nbr  mean_sales  total_sales  zero_sales_share  n_rows
       13  189.358775 1.052305e+07          0.331570   55572
       20  232.508360 1.292095e+07          0.544717   55572
       45  980.673908 5.449801e+07          0.193587   55572
       49  781.330450 4.342010e+07          0.193479   55572
       50  515.601753 2.865302e+07          0.206525   55572
fold_1: train through 2017-06-18, validate 2017-06-19 to 2017-07-02
fold_2: train through 2017-07-02, validate 2017-07-03 to 2017-07-16
fold_3: train through 2017-07-16, validate 2017-07-17 to 2017-07-30


In [30]:

# STEP 6A —  for one LightGBM configuration

import time
import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import ParameterSampler

SEED = 42
HORIZON = 14

lag_cols = [
    "lag_1", "lag_7", "lag_14", "lag_28", "lag_364",
    "rolling_mean_7", "rolling_mean_28",
    "rolling_mean_56", "rolling_std_7",
]

categorical_cols = [
    "store_nbr", "family", "city", "state", "type", "cluster"
]

feature_cols = [
    c for c in model_df.columns
    if c not in ["id", "date", "sales"]
]

# Ensure the selected stores and fold definitions already exist.
tune_df = model_df.loc[
    model_df["store_nbr"].isin(selected_stores)
].sort_values(["date", "store_nbr", "family"]).copy()


def recursive_fold_score(params, fold_start, fold_end):
    train_end = fold_start - pd.Timedelta(days=1)

    train_part = tune_df.loc[tune_df["date"] <= train_end].copy()
    valid_part = tune_df.loc[
        tune_df["date"].between(fold_start, fold_end)
    ].copy()

    # Fit only on historical rows.
    model = lgb.LGBMRegressor(
        objective="regression_l1",
        n_jobs=-1,
        random_state=SEED,
        verbosity=-1,
        subsample_freq=1,
        **params,
    )

    model.fit(
        train_part[feature_cols],
        train_part["sales"],
        categorical_feature=categorical_cols,
    )

    # Histories contain only observed sales through train_end.
    histories = {}
    for (store_id, family), group in train_part.groupby(
        ["store_nbr", "family"], observed=True, sort=False
    ):
        histories[(int(store_id), str(family))] = (
            group.sort_values("date")["sales"].astype(float).tolist()
        )

    predictions = []
    actuals = []
    store_ids = []

    for day in pd.date_range(fold_start, fold_end, freq="D"):
        day_rows = valid_part.loc[valid_part["date"].eq(day)]

        records = []
        keys = []

        for _, row in day_rows.iterrows():
            key = (int(row["store_nbr"]), str(row["family"]))
            history = histories[key]

            # Non-lag features are known calendar/store/promotion inputs.
            record = {c: row[c] for c in feature_cols if c not in lag_cols}

            record["lag_1"] = history[-1] if len(history) >= 1 else np.nan
            record["lag_7"] = history[-7] if len(history) >= 7 else np.nan
            record["lag_14"] = history[-14] if len(history) >= 14 else np.nan
            record["lag_28"] = history[-28] if len(history) >= 28 else np.nan
            record["lag_364"] = history[-364] if len(history) >= 364 else np.nan

            record["rolling_mean_7"] = (
                np.mean(history[-7:]) if len(history) >= 7 else np.nan
            )
            record["rolling_mean_28"] = (
                np.mean(history[-28:]) if len(history) >= 28 else np.nan
            )
            record["rolling_mean_56"] = (
                np.mean(history[-56:]) if len(history) >= 56 else np.nan
            )
            record["rolling_std_7"] = (
                np.std(history[-7:], ddof=1)
                if len(history) >= 7 else np.nan
            )

            records.append(record)
            keys.append(key)

        X_day = pd.DataFrame(records, columns=feature_cols)

        # Preserve the exact category levels used during model fitting.
        for col in categorical_cols:
            X_day[col] = pd.Categorical(
                X_day[col],
                categories=model_df[col].cat.categories,
            )

        day_predictions = np.maximum(model.predict(X_day), 0)

        # Score against actual sales, but never feed actuals into histories.
        predictions.extend(day_predictions)
        actuals.extend(day_rows["sales"].to_numpy())
        store_ids.extend(day_rows["store_nbr"].astype(int).to_numpy())

        # Append predictions only after all forecasts for this day are made.
        for key, pred in zip(keys, day_predictions):
            histories[key].append(float(pred))

    actuals = np.asarray(actuals, dtype=float)
    predictions = np.asarray(predictions, dtype=float)

    wape = (
        np.abs(actuals - predictions).sum() / np.abs(actuals).sum()
        if np.abs(actuals).sum() > 0 else np.nan
    )
    mae = np.abs(actuals - predictions).mean()

    return {
        "wape": wape,
        "mae": mae,
        "n_predictions": len(actuals),
    }


## Run the 20-trial randomized search

In [31]:

# STEP 7A — Randomized hyperparameter search (20 trials x 3 folds)

param_dist = {
    "n_estimators": randint(200, 601),
    "learning_rate": loguniform(0.015, 0.12),
    "num_leaves": randint(15, 64),
    "max_depth": [-1, 6, 8, 10, 12],
    "min_child_samples": randint(50, 501),
    "subsample": uniform(0.65, 0.35),
    "colsample_bytree": uniform(0.65, 0.35),
    "reg_alpha": loguniform(0.001, 10),
    "reg_lambda": loguniform(0.01, 100),
    "min_split_gain": uniform(0, 0.2),
}

sampled_params = list(
    ParameterSampler(param_dist, n_iter=20, random_state=SEED)
)

search_results = []

for trial, params in enumerate(sampled_params, start=1):
    fold_results = []
    start_time = time.perf_counter()

    for fold_name, fold_start, fold_end in folds:
        scores = recursive_fold_score(
            params=params,
            fold_start=fold_start,
            fold_end=fold_end,
        )
        fold_results.append(scores)

    elapsed = time.perf_counter() - start_time

    result = {
        "trial": trial,
        "mean_wape": np.mean([r["wape"] for r in fold_results]),
        "mean_mae": np.mean([r["mae"] for r in fold_results]),
        "elapsed_seconds": elapsed,
        **params,
    }

    search_results.append(result)

    print(
        f"Trial {trial:02d}/20 | "
        f"WAPE: {result['mean_wape']:.4f} | "
        f"MAE: {result['mean_mae']:.3f} | "
        f"Time: {elapsed:.1f}s"
    )

search_results_df = (
    pd.DataFrame(search_results)
    .sort_values(["mean_wape", "mean_mae"])
    .reset_index(drop=True)
)

print("\nTop 5 configurations:")
print(search_results_df.head(5).to_string(index=False))

best_params = {
    key: search_results_df.loc[0, key]
    for key in param_dist
}

# Convert values back to their expected Python types.
best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["num_leaves"] = int(best_params["num_leaves"])
best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_child_samples"] = int(best_params["min_child_samples"])

print("\nBest parameters:")
print(best_params)


Trial 01/20 | WAPE: 0.1014 | MAE: 73.619 | Time: 6.3s
Trial 02/20 | WAPE: 0.1050 | MAE: 76.255 | Time: 7.6s
Trial 03/20 | WAPE: 0.1003 | MAE: 72.824 | Time: 7.7s
Trial 04/20 | WAPE: 0.0975 | MAE: 70.779 | Time: 6.8s
Trial 05/20 | WAPE: 0.0962 | MAE: 69.839 | Time: 6.3s
Trial 06/20 | WAPE: 0.0987 | MAE: 71.636 | Time: 11.2s
Trial 07/20 | WAPE: 0.1031 | MAE: 74.849 | Time: 8.4s
Trial 08/20 | WAPE: 0.0896 | MAE: 64.979 | Time: 12.2s
Trial 09/20 | WAPE: 0.0937 | MAE: 68.004 | Time: 7.3s
Trial 10/20 | WAPE: 0.0888 | MAE: 64.402 | Time: 11.1s
Trial 11/20 | WAPE: 0.0933 | MAE: 67.728 | Time: 7.6s
Trial 12/20 | WAPE: 0.1017 | MAE: 73.804 | Time: 5.4s
Trial 13/20 | WAPE: 0.0923 | MAE: 66.939 | Time: 9.5s
Trial 14/20 | WAPE: 0.0943 | MAE: 68.425 | Time: 6.9s
Trial 15/20 | WAPE: 0.0905 | MAE: 65.593 | Time: 12.5s
Trial 16/20 | WAPE: 0.0908 | MAE: 65.858 | Time: 13.9s
Trial 17/20 | WAPE: 0.0992 | MAE: 71.978 | Time: 5.4s
Trial 18/20 | WAPE: 0.1354 | MAE: 98.392 | Time: 6.5s
Trial 19/20 | WAPE: 0.1

## Train and evaluate the four models

In [32]:

import time
import numpy as np
import pandas as pd
import lightgbm as lgb

# Final architecture-comparison split
validation_start = pd.Timestamp("2017-07-31")
cutoff_date = validation_start - pd.Timedelta(days=1)
validation_end = pd.Timestamp("2017-08-15")

fit_df = model_df.loc[model_df["date"] <= cutoff_date].copy()
valid_df = (
    model_df.loc[
        model_df["date"].between(validation_start, validation_end)
    ]
    .sort_values(["date", "store_nbr", "family"])
    .copy()
)

lag_cols = [
    "lag_1", "lag_7", "lag_14", "lag_28", "lag_364",
    "rolling_mean_7", "rolling_mean_28",
    "rolling_mean_56", "rolling_std_7",
]

all_features = [
    c for c in model_df.columns
    if c not in ["id", "date", "sales"]
]

global_features = all_features
cluster_features = [c for c in all_features if c != "cluster"]
store_features = [
    c for c in all_features
    if c not in ["store_nbr", "cluster", "city", "state", "type"]
]
store_family_features = [
    c for c in all_features
    if c not in [
        "store_nbr", "family", "cluster", "city", "state", "type"
    ]
]

feature_sets = {
    "global": global_features,
    "cluster": cluster_features,
    "store": store_features,
    "store_family": store_family_features,
}

# Recover parameters directly from the winning search result.
best_row = search_results_df.sort_values("mean_wape").iloc[0]
best_params = {
    key: best_row[key]
    for key in [
        "n_estimators", "learning_rate", "num_leaves", "max_depth",
        "min_child_samples", "subsample", "colsample_bytree",
        "reg_alpha", "reg_lambda", "min_split_gain",
    ]
}

for key in ["n_estimators", "num_leaves", "max_depth", "min_child_samples"]:
    best_params[key] = int(best_params[key])

print("Training rows:", len(fit_df))
print("Holdout rows:", len(valid_df))
print("Holdout dates:", valid_df["date"].min(), "to", valid_df["date"].max())
print("Parameters:", best_params)

def fit_lgbm(data, features, n_jobs=-1):
    categorical_features = [
        col for col in features
        if isinstance(data[col].dtype, pd.CategoricalDtype)
    ]

    model = lgb.LGBMRegressor(
        objective="regression_l1",
        random_state=42,
        verbosity=-1,
        n_jobs=n_jobs,
        subsample_freq=1,
        **best_params,
    )

    model.fit(
        data[features],
        data["sales"],
        categorical_feature=categorical_features,
    )
    return model

models = {}
training_times = {}

# 1. Global: one model across all series
start = time.perf_counter()
models["global"] = fit_lgbm(fit_df, global_features)
training_times["global"] = time.perf_counter() - start
print(f"Global model trained in {training_times['global']:.1f}s")

# 2. Cluster: one model per cluster
start = time.perf_counter()
models["cluster"] = {
    int(cluster_id): fit_lgbm(group, cluster_features)
    for cluster_id, group in fit_df.groupby("cluster", observed=True)
}
training_times["cluster"] = time.perf_counter() - start
print(f"Cluster models trained: {len(models['cluster'])}")

# 3. Store: one model per store
start = time.perf_counter()
models["store"] = {
    int(store_id): fit_lgbm(group, store_features)
    for store_id, group in fit_df.groupby("store_nbr", observed=True)
}
training_times["store"] = time.perf_counter() - start
print(f"Store models trained: {len(models['store'])}")

# 4. Store-family: one model per individual series
# Use one thread per small model to avoid CPU oversubscription.
start = time.perf_counter()
models["store_family"] = {
    (int(store_id), str(family)): fit_lgbm(
        group, store_family_features, n_jobs=1
    )
    for (store_id, family), group in fit_df.groupby(
        ["store_nbr", "family"], observed=True
    )
}
training_times["store_family"] = time.perf_counter() - start
print(f"Store-family models trained: {len(models['store_family'])}")

print("\nTraining times:")
for name, seconds in training_times.items():
    print(f"{name}: {seconds:.1f}s")


Training rows: 2972376
Holdout rows: 28512
Holdout dates: 2017-07-31 00:00:00 to 2017-08-15 00:00:00
Parameters: {'n_estimators': 467, 'learning_rate': 0.05808612452480186, 'num_leaves': 48, 'max_depth': 10, 'min_child_samples': 184, 'subsample': 0.8840944384322625, 'colsample_bytree': 0.9742053074796722, 'reg_alpha': 0.002407911195464457, 'reg_lambda': 0.3042797205957134, 'min_split_gain': 0.17000771555795988}
Global model trained in 22.8s
Cluster models trained: 17
Store models trained: 54
Store-family models trained: 1782

Training times:
global: 22.8s
cluster: 40.9s
store: 71.9s
store_family: 107.1s


In [33]:

# Recursive 16-day forecast evaluation

categorical_cols = [
    "store_nbr", "family", "city", "state", "type", "cluster"
]

def evaluate_strategy(strategy):
    features = feature_sets[strategy]

    # Histories contain observed sales through the cutoff only.
    histories = {
        (int(store_id), str(family)): (
            group.sort_values("date")["sales"].astype(float).tolist()
        )
        for (store_id, family), group in fit_df.groupby(
            ["store_nbr", "family"], observed=True, sort=False
        )
    }

    all_predictions = []
    all_actuals = []
    all_stores = []
    all_families = []
    all_dates = []

    non_lag_features = [c for c in all_features if c not in lag_cols]

    for day in pd.date_range(validation_start, validation_end, freq="D"):
        day_rows = valid_df.loc[valid_df["date"].eq(day)].copy()

        records = []
        keys = []

        for _, row in day_rows.iterrows():
            key = (int(row["store_nbr"]), str(row["family"]))
            history = histories[key]

            record = {c: row[c] for c in non_lag_features}
            record["lag_1"] = history[-1] if len(history) >= 1 else np.nan
            record["lag_7"] = history[-7] if len(history) >= 7 else np.nan
            record["lag_14"] = history[-14] if len(history) >= 14 else np.nan
            record["lag_28"] = history[-28] if len(history) >= 28 else np.nan
            record["lag_364"] = history[-364] if len(history) >= 364 else np.nan

            record["rolling_mean_7"] = (
                np.mean(history[-7:]) if len(history) >= 7 else np.nan
            )
            record["rolling_mean_28"] = (
                np.mean(history[-28:]) if len(history) >= 28 else np.nan
            )
            record["rolling_mean_56"] = (
                np.mean(history[-56:]) if len(history) >= 56 else np.nan
            )
            record["rolling_std_7"] = (
                np.std(history[-7:], ddof=1)
                if len(history) >= 7 else np.nan
            )

            records.append(record)
            keys.append(key)

        X_day = pd.DataFrame(records, columns=all_features)

        for col in categorical_cols:
            X_day[col] = pd.Categorical(
                X_day[col],
                categories=model_df[col].cat.categories,
            )

        X = X_day[features]
        predictions = np.empty(len(day_rows), dtype=float)

        if strategy == "global":
            predictions = models["global"].predict(X)

        elif strategy == "cluster":
            for cluster_id, positions in day_rows.groupby(
                "cluster", observed=True
            ).indices.items():
                positions = np.asarray(positions, dtype=int)
                predictions[positions] = models["cluster"][
                    int(cluster_id)
                ].predict(X.iloc[positions])

        elif strategy == "store":
            for store_id, positions in day_rows.groupby(
                "store_nbr", observed=True
            ).indices.items():
                positions = np.asarray(positions, dtype=int)
                predictions[positions] = models["store"][
                    int(store_id)
                ].predict(X.iloc[positions])

        elif strategy == "store_family":
            for i, key in enumerate(keys):
                predictions[i] = models["store_family"][key].predict(
                    X.iloc[[i]]
                )[0]

        predictions = np.maximum(predictions, 0)

        all_predictions.extend(predictions)
        all_actuals.extend(day_rows["sales"].to_numpy())
        all_stores.extend(day_rows["store_nbr"].astype(int).to_numpy())
        all_families.extend(day_rows["family"].astype(str).to_numpy())
        all_dates.extend(day_rows["date"].to_numpy())

        # Append forecasts only after the whole day is predicted.
        for key, pred in zip(keys, predictions):
            histories[key].append(float(pred))

    result = pd.DataFrame({
        "date": all_dates,
        "store_nbr": all_stores,
        "family": all_families,
        "actual": all_actuals,
        "prediction": all_predictions,
    })

    error = np.abs(result["actual"] - result["prediction"])
    wape = error.sum() / result["actual"].abs().sum()
    mae = error.mean()

    return result, {
        "strategy": strategy,
        "n_models": (
            len(models[strategy])
            if isinstance(models[strategy], dict) else 1
        ),
        "wape": wape,
        "mae": mae,
        "training_seconds": training_times[strategy],
    }

forecast_results = {}
summary_rows = []

for strategy in ["global", "cluster", "store", "store_family"]:
    start = time.perf_counter()
    forecast_df, metrics = evaluate_strategy(strategy)
    metrics["evaluation_seconds"] = time.perf_counter() - start

    forecast_results[strategy] = forecast_df
    summary_rows.append(metrics)

comparison_df = (
    pd.DataFrame(summary_rows)
    .sort_values(["wape", "mae"])
    .reset_index(drop=True)
)

print(comparison_df.to_string(index=False))


    strategy  n_models     wape       mae  training_seconds  evaluation_seconds
     cluster        17 0.151316 70.686350         40.933751            2.511372
       store        54 0.153437 71.676914         71.934707            2.905476
      global         1 0.166511 77.784613         22.754713            2.240810
store_family      1782 0.185657 86.728578        107.120342            9.842913


### Two checks before we lock the model


First, check whether the cluster model performs consistently across stores and product families. An aggregate WAPE can hide poor performance on particular series.




In [34]:

# STEP 11A — Cluster-model performance by store and family

cluster_predictions = forecast_results["cluster"].copy()

cluster_predictions["abs_error"] = (
    cluster_predictions["actual"] - cluster_predictions["prediction"]
).abs()

def grouped_metrics(df, group_col):
    result = (
        df.groupby(group_col, observed=True)
        .agg(
            rows=("actual", "size"),
            actual_sales=("actual", "sum"),
            MAE=("abs_error", "mean"),
            total_abs_error=("abs_error", "sum"),
        )
        .reset_index()
    )

    result["WAPE"] = (
        result["total_abs_error"] / result["actual_sales"].abs()
    )

    return result.sort_values("WAPE", ascending=False)

# Add cluster IDs for diagnostics
store_cluster_map = (
    model_df[["store_nbr", "cluster"]]
    .drop_duplicates()
    .copy()
)

cluster_predictions["store_nbr"] = (
    cluster_predictions["store_nbr"].astype("int64")
)

cluster_predictions = cluster_predictions.merge(
    store_cluster_map.assign(
        store_nbr=store_cluster_map["store_nbr"].astype("int64")
    ),
    on="store_nbr",
    how="left",
    validate="many_to_one",
)

print("Worst 10 stores by WAPE:")
print(grouped_metrics(cluster_predictions, "store_nbr").head(10).to_string(index=False))

print("\nWorst 10 families by WAPE:")
print(grouped_metrics(cluster_predictions, "family").head(10).to_string(index=False))

print("\nCluster-level WAPE:")
print(grouped_metrics(cluster_predictions, "cluster").to_string(index=False))


Worst 10 stores by WAPE:
 store_nbr  rows  actual_sales        MAE  total_abs_error     WAPE
        26   528  74602.877001  39.264112     20731.451261 0.277891
        32   528  83639.456997  39.861155     21046.689902 0.251636
        38   528 197783.477011  88.227646     46584.197257 0.235531
        25   528 154921.537115  68.514316     36175.559059 0.233509
        30   528 105535.491004  44.774886     23641.139751 0.224011
        22   528 121737.915980  51.468618     27175.430132 0.223229
        54   528 176572.478994  73.197671     38648.370043 0.218881
        39   528 261722.187096 108.153851     57105.233577 0.218190
        14   528 136648.141012  54.720314     28892.325793 0.211436
        40   528 287750.831000 114.312547     60357.024617 0.209754

Worst 10 families by WAPE:
                    family  rows  actual_sales       MAE  total_abs_error      WAPE
                     BOOKS   864           9.0  0.620396       536.021887 59.557987
                 BABY CARE   86

## Select and validate the hybrid architecture

In [36]:
# 1. Copy predictions from both models
cluster_pred = forecast_results["cluster"].copy()
store_pred = forecast_results["store"].copy()

keys = ["date", "store_nbr", "family"]

# 2. Merge predictions; keep actual only once
comparison = cluster_pred[
    keys + ["actual", "prediction"]
].merge(
    store_pred[keys + ["prediction"]],
    on=keys,
    suffixes=("_cluster", "_store"),
    validate="one_to_one",
)

# 3. Validate alignment
assert len(comparison) == len(cluster_pred) == len(store_pred)
assert comparison["actual"].notna().all()

# 4. Calculate absolute errors
comparison["error_cluster"] = (
    comparison["actual"] - comparison["prediction_cluster"]
).abs()

comparison["error_store"] = (
    comparison["actual"] - comparison["prediction_store"]
).abs()

# 5. Compare models by store
store_comparison = (
    comparison.groupby("store_nbr", observed=True)
    .agg(
        actual_sales=("actual", "sum"),
        cluster_abs_error=("error_cluster", "sum"),
        store_abs_error=("error_store", "sum"),
        cluster_MAE=("error_cluster", "mean"),
        store_MAE=("error_store", "mean"),
        rows=("actual", "size"),
    )
    .reset_index()
)

# Avoid division by zero for stores with zero total actual sales
store_comparison["cluster_WAPE"] = (
    store_comparison["cluster_abs_error"]
    / store_comparison["actual_sales"].abs().replace(0, np.nan)
)

store_comparison["store_WAPE"] = (
    store_comparison["store_abs_error"]
    / store_comparison["actual_sales"].abs().replace(0, np.nan)
)

# Positive improvement means the individual store model is better
store_comparison["store_improvement_pp"] = 100 * (
    store_comparison["cluster_WAPE"]
    - store_comparison["store_WAPE"]
)

store_comparison["store_wins"] = (
    store_comparison["store_improvement_pp"] > 0
)

# 6. Add each store's cluster
store_cluster_map = (
    model_df[["store_nbr", "cluster"]]
    .drop_duplicates()
    .copy()
)

store_cluster_map["store_nbr"] = (
    store_cluster_map["store_nbr"].astype("int64")
)

store_comparison["store_nbr"] = (
    store_comparison["store_nbr"].astype("int64")
)

store_comparison = store_comparison.merge(
    store_cluster_map,
    on="store_nbr",
    how="left",
    validate="one_to_one",
)

# 7. Display results
display(
    store_comparison.sort_values(
        "store_improvement_pp", ascending=False
    ).round(4)
)

print(
    "Stores where individual model wins:",
    int(store_comparison["store_wins"].sum()),
)

print(
    "Stores where cluster model wins:",
    int((~store_comparison["store_wins"]).sum()),
)

# 8. Aggregate comparison by cluster
cluster_comparison = (
    comparison.merge(
        store_cluster_map,
        on="store_nbr",
        how="left",
        validate="many_to_one",
    )
    .groupby("cluster", observed=True)
    .agg(
        actual_sales=("actual", "sum"),
        cluster_abs_error=("error_cluster", "sum"),
        store_abs_error=("error_store", "sum"),
        rows=("actual", "size"),
    )
    .reset_index()
)

cluster_comparison["cluster_WAPE"] = (
    cluster_comparison["cluster_abs_error"]
    / cluster_comparison["actual_sales"].abs().replace(0, np.nan)
)

cluster_comparison["store_WAPE"] = (
    cluster_comparison["store_abs_error"]
    / cluster_comparison["actual_sales"].abs().replace(0, np.nan)
)

cluster_comparison["store_improvement_pp"] = 100 * (
    cluster_comparison["cluster_WAPE"]
    - cluster_comparison["store_WAPE"]
)

display(
    cluster_comparison.sort_values(
        "store_improvement_pp", ascending=False
    ).round(4)
)


,store_nbr,actual_sales,cluster_abs_error,store_abs_error,cluster_MAE,store_MAE,rows,cluster_WAPE,store_WAPE,store_improvement_pp,store_wins,cluster
25,26,74602.8770,20731.4513,15568.5072,39.2641,29.4858,528,0.2779,0.2087,6.9206,True,10
49,50,326493.2400,65465.4568,53902.1741,123.9876,102.0875,528,0.2005,0.1651,3.5417,True,14
0,1,155135.6292,25386.4044,22790.8353,48.0803,43.1645,528,0.1636,0.1469,1.6731,True,13
24,25,154921.5371,36175.5591,33961.1927,68.5143,64.3204,528,0.2335,0.2192,1.4293,True,1
36,37,196512.9110,35342.3475,32640.7438,66.9363,61.8196,528,0.1798,0.1661,1.3748,True,2
4,5,156442.8789,22959.4794,20827.8235,43.4839,39.4466,528,0.1468,0.1331,1.3626,True,4
30,31,236023.3811,34597.3504,31706.1587,65.5253,60.0495,528,0.1466,0.1343,1.2250,True,10
21,22,121737.9160,27175.4301,25723.8444,51.4686,48.7194,528,0.2232,0.2113,1.1924,True,7
15,16,104594.0920,19656.0356,18417.9222,37.2273,34.8824,528,0.1879,0.1761,1.1837,True,3
3,4,187479.3410,28090.1475,25927.5156,53.2010,49.1051,528,0.1498,0.1383,1.1535,True,9


Stores where individual model wins: 29
Stores where cluster model wins: 25


,cluster,actual_sales,cluster_abs_error,store_abs_error,rows,cluster_WAPE,store_WAPE,store_improvement_pp
8,9,3.078658e+05,48403.7387,46108.1090,1056,0.1572,0.1498,0.7457
0,1,6.597215e+05,94720.8849,89958.2541,1584,0.1436,0.1364,0.7219
13,14,1.768793e+06,298788.5768,287944.2402,2112,0.1689,0.1628,0.6131
7,8,1.149151e+06,132987.1516,126376.0344,1584,0.1157,0.1100,0.5753
6,7,2.583861e+05,56067.7559,54940.3272,1056,0.2170,0.2126,0.4363
9,10,1.262108e+06,197162.1634,192174.3510,3168,0.1562,0.1523,0.3952
12,13,8.052472e+05,112843.1622,111022.8359,2112,0.1401,0.1379,0.2261
1,2,3.781806e+05,71770.3273,71488.5155,1056,0.1898,0.1890,0.0745
4,5,6.566514e+05,100125.1550,100125.1550,528,0.1525,0.1525,0.0000
11,12,2.208803e+05,31799.6182,31799.6182,528,0.1440,0.1440,0.0000


Keep the cluster model as our default architecture for now

# Forecasting model

Train and save the final cluster models

In [37]:

from pathlib import Path
import joblib
import time

# 1. Final model configuration selected during tuning
FINAL_PARAMS = {
    "n_estimators": 467,
    "learning_rate": 0.05808612452480186,
    "num_leaves": 48,
    "max_depth": 10,
    "min_child_samples": 184,
    "subsample": 0.8840944384322625,
    "colsample_bytree": 0.9742053074796722,
    "reg_alpha": 0.002407911195464457,
    "reg_lambda": 0.3042797205957134,
    "min_split_gain": 0.17000771555795988,
}

# 2. Define the model's input features
feature_cols = [
    col for col in model_df.columns
    if col not in ["id", "date", "sales", "cluster"]
]

categorical_cols = [
    col for col in feature_cols
    if isinstance(model_df[col].dtype, pd.CategoricalDtype)
]

# 3. Train one model per cluster
final_cluster_models = {}
start_time = time.time()

cluster_ids = sorted(model_df["cluster"].dropna().unique())

for cluster_id in cluster_ids:
    cluster_data = model_df.loc[
        model_df["cluster"] == cluster_id
    ]

    X = cluster_data[feature_cols]
    y = cluster_data["sales"]

    model = lgb.LGBMRegressor(
        **FINAL_PARAMS,
        objective="regression_l1",
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
        subsample_freq=1,
    )

    model.fit(
        X,
        y,
        categorical_feature=categorical_cols,
    )

    final_cluster_models[int(cluster_id)] = model

    print(
        f"Cluster {cluster_id}: "
        f"{len(cluster_data):,} rows | model trained"
    )

print(f"\nModels trained: {len(final_cluster_models)}")
print(f"Total training time: {(time.time() - start_time):.1f} seconds")

# 4. Save model artifacts and the information needed for inference
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    "models": final_cluster_models,
    "feature_cols": feature_cols,
    "categorical_cols": categorical_cols,
    "category_levels": {
        col: list(model_df[col].cat.categories)
        for col in categorical_cols
    },
    "cluster_ids": [int(c) for c in cluster_ids],
    "last_train_date": model_df["date"].max(),
    "history": model_df[
        ["date", "store_nbr", "family", "sales", "onpromotion"]
    ].copy(),
    "stores": stores.copy(),
    "holidays_events": holidays_events.copy(),
}

artifact_path = ARTIFACT_DIR / "sales_forecasting_artifact.joblib"
joblib.dump(artifact, artifact_path, compress=3)

print(f"Saved artifact: {artifact_path}")
print(f"Artifact size: {artifact_path.stat().st_size / 1e6:.1f} MB")


Cluster 1: 166,716 rows | model trained
Cluster 2: 111,144 rows | model trained
Cluster 3: 389,004 rows | model trained
Cluster 4: 166,716 rows | model trained
Cluster 5: 55,572 rows | model trained
Cluster 6: 333,432 rows | model trained
Cluster 7: 111,144 rows | model trained
Cluster 8: 166,716 rows | model trained
Cluster 9: 111,144 rows | model trained
Cluster 10: 333,432 rows | model trained
Cluster 11: 166,716 rows | model trained
Cluster 12: 55,572 rows | model trained
Cluster 13: 222,288 rows | model trained
Cluster 14: 222,288 rows | model trained
Cluster 15: 277,860 rows | model trained
Cluster 16: 55,572 rows | model trained
Cluster 17: 55,572 rows | model trained

Models trained: 17
Total training time: 35.6 seconds
Saved artifact: artifacts/sales_forecasting_artifact.joblib
Artifact size: 19.1 MB


# Good work! models are trained

 Next is setup the Envn

## Enviroment Setup

In [38]:

import sys
import pandas as pd
import numpy as np
import lightgbm as lgb
import sklearn
import joblib

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("LightGBM:", lgb.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)


Python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]
Python executable: /opt/anaconda3/bin/python
pandas: 2.2.2
NumPy: 1.26.4
LightGBM: 4.5.0
scikit-learn: 1.5.1
joblib: 1.4.2


In [ ]:
# ## Mac terminal:

# python3 --version
# git --version
# node --version
# npm --version


# ## #Init Git


# git init

# cat > .gitignore <<'EOF'
# <!-- # Python -->
# __pycache__/
# *.py[cod]
# .pytest_cache/
# .venv/
# **/.venv/

# <!-- # Secrets and local configuration -->
# .env
# .env.*
# !.env.example

# <!-- # Node / Next.js -->
# node_modules/
# .next/
# out/

# <!-- # OS -->
# .DS_Store

# <!-- # Data and temporary files -->
# data/raw/
# data/processed/
# EOF

# git status --short




In [ ]:
# # create a requirements.txt file for the environment


# cat > backend/requirements.txt <<'EOF'
# numpy==1.26.4
# pandas==2.2.2
# lightgbm==4.5.0
# joblib==1.4.2
# fastapi==0.115.12
# uvicorn[standard]==0.34.2
# pydantic==2.11.4
# pytest==8.3.5
# httpx==0.28.1
# EOF



# create isolated environment and install dependencies


# /opt/anaconda3/bin/python -m venv backend/.venv

# backend/.venv/bin/python -m pip install --upgrade pip

# backend/.venv/bin/python -m pip install -r backend/requirements.txt



# validate the environment and artifact

# backend/.venv/bin/python -c "import sys, numpy, pandas, lightgbm, joblib, fastapi, pydantic; print('Python:', sys.version.split()[0]); print('NumPy:', numpy.__version__); print('pandas:', pandas.__version__); print('LightGBM:', lightgbm.__version__); print('joblib:', joblib.__version__); print('FastAPI:', fastapi.__version__); print('Pydantic:', pydantic.__version__)"

# find backend frontend -maxdepth 3 -type f | sort

# git status --short


# Forecasting engine

In [39]:

## validate the saved artifact

from pathlib import Path
import joblib

artifact_path = Path("artifacts/sales_forecasting_artifact.joblib")
assert artifact_path.exists(), f"Artifact not found: {artifact_path}"

artifact = joblib.load(artifact_path)

print("Artifact loaded successfully")
print("Artifact keys:", list(artifact.keys()))
print("Number of models:", len(artifact["models"]))
print("Cluster IDs:", sorted(artifact["models"].keys()))
print("Number of features:", len(artifact["feature_cols"]))
print("Features:", artifact["feature_cols"])
print("Categorical features:", artifact["categorical_cols"])

history = artifact["history"]
print("\nHistorical data shape:", history.shape)
print("History date range:", history["date"].min(), "to", history["date"].max())
print("Stores:", history["store_nbr"].nunique())
print("Product families:", history["family"].nunique())
print("Duplicate date-store-family rows:",
      history.duplicated(["date", "store_nbr", "family"]).sum())

print("\nStore metadata columns:", list(artifact["stores"].columns))
print("Holiday data columns:", list(artifact["holidays_events"].columns))

# Verify the saved models expect the same feature schema
model = artifact["models"][sorted(artifact["models"].keys())[0]]
print("\nModel feature schema matches:",
      model.feature_name_ == artifact["feature_cols"])



Artifact loaded successfully
Artifact keys: ['models', 'feature_cols', 'categorical_cols', 'category_levels', 'cluster_ids', 'last_train_date', 'history', 'stores', 'holidays_events']
Number of models: 17
Cluster IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Number of features: 25
Features: ['store_nbr', 'family', 'onpromotion', 'city', 'state', 'type', 'day_of_week', 'day_of_month', 'month', 'year', 'is_weekend', 'holiday_event_count', 'has_national_event', 'has_regional_event', 'has_local_event', 'has_transferred_event', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_364', 'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_56', 'rolling_std_7']
Categorical features: ['store_nbr', 'family', 'city', 'state', 'type']

Historical data shape: (3000888, 5)
History date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
Stores: 54
Product families: 33
Duplicate date-store-family rows: 0

Store metadata columns: ['store_nbr', 'city', 'state', 'type', 'cluster']
Holiday data col

## Implement the forecasting engine

- Create backend/app/forecasting.py
- Test the forecasting engine

<!-- 
backend/.venv/bin/python - <<'PY'
import joblib
import numpy as np
from pathlib import Path
from backend.app.forecasting import forecast_sales

artifact = joblib.load(
    Path("artifacts/sales_forecasting_artifact.joblib")
)

forecast = forecast_sales(
    artifact=artifact,
    store_nbr=1,
    family="BEVERAGES",
    start_date="2017-08-16",
    horizon=7,
    assume_zero_promotions=True,
)

print(forecast.to_string(index=False))

assert len(forecast) == 7
assert forecast["date"].is_unique
assert np.isfinite(forecast["predicted_sales"]).all()
assert (forecast["predicted_sales"] >= 0).all()
assert forecast["promotion_assumption"].eq("zero_promotions").all()

print("\nPASS: 7 daily forecasts generated successfully.")
PY -->


## Milestone 4 is complete: the forecasting engine produces valid 7-day predictions.

# 5 — Build the FastAPI backend

Step 
1. — Create backend/app/main.py

2. start server
PYTHONPATH=. backend/.venv/bin/python -m uvicorn backend.app.main:app --reload --port 8000

3. Test the API
http://localhost:8000/docs#/default/create_forecast_forecast_post

## Milestone 4: completed

# 6: Next.js frontend

Next.js App Router for the application structure.



TypeScript to catch mistakes in request and response data.



Tailwind CSS for styling quickly.



Recharts for the forecast chart.

1. setup

cd frontend
npx create-next-app@latest . --typescript --eslint --tailwind --app --src-dir --use-npm --import-alias "@/*"


2.  Install the chart library


3. updated page.tsx